# Ramanujan's $\tau(n)$ and the Sato–Tate law

*maths · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/ramanujan_tau)

Expand the discriminant modular form $\Delta(q) = q\prod_n (1-q^n)^{24}$ as a tensor power series to get Ramanujan's $\tau(n)$, exactly to $n \sim 10^5$–$10^6$ through residues modulo a batch of primes. Check that $\tau$ is multiplicative and obeys the Hecke recurrence, that $\tau(n) \equiv \sigma_{11}(n) \pmod{691}$, that $|\tau(p)| \le 2p^{11/2}$ (Deligne's theorem, 1974), and watch the normalised values $\tau(p)/2p^{11/2}$ fill in the Sato–Tate semicircle (a theorem for $\Delta$ since 2011). The program checks these theorems numerically.

**Vector form**

$$\Delta(q) = q\prod_{n\ge1}(1-q^n)^{24} = \sum_{n\ge1}\tau(n)\,q^n,\qquad \tau(p^{k+1}) = \tau(p)\tau(p^k) - p^{11}\tau(p^{k-1}),\qquad |\tau(p)| \le 2p^{11/2},\qquad x_p = \frac{\tau(p)}{2p^{11/2}} \sim \frac{2}{\pi}\sqrt{1-x^2}\,dx$$

**Index (tensor) form**

$$e^{(3)}_n = \sum_k (-1)^k(2k+1)\,\delta_{n,\,k(k+1)/2},\qquad e^{(2a)}_n = \sum_j e^{(a)}_j\, e^{(a)}_{n-j},\qquad \tau_{n+1} = e^{(24)}_n,\qquad r_{mn} = \Big(\sum_j r^{(a)}_{mj}\, r^{(a)}_{m,n-j}\Big) \bmod \mu_m$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $e^{(3)}_n$ | `eta_cubed(n_max, dev, dtype)` | (N,) | Jacobi's identity $\prod(1-q^n)^3 = \sum_k (-1)^k (2k+1) q^{k(k+1)/2}$: only $O(\sqrt N)$ nonzero coefficients |
| $e^{(2a)}_n = \sum_j e^{(a)}_j e^{(a)}_{n-j}$ | `toeplitz @ b` | (N, N) @ (N,) | repeated $j$ ⇒ summed: a truncated series product as a Toeplitz matrix times a vector, exact in int64 for small $N$ |
| $e^{(3)} \to e^{(6)} \to e^{(12)} \to e^{(24)}$ | `for _ in range(3): e = mul(e, e)` | (N,) | three squarings give the 24th power |
| $r_{mn}$ | `irfft(rfft(a) * rfft(b)).round() % m` | (M, N) | the same products by FFT for $M$ prime moduli at once, rounded to exact residues (each $m^2 N \le 2^{42}$) |
| $\tau(n)$ | `garner_balanced(residues.T, moduli)` | (N, M) | Chinese remainder theorem with balanced digits, so negative values come out negative |
| $\tau(mn) = \tau(m)\tau(n)$ | `tau[m * n] == tau[m] * tau[n]` | (pairs,) | every coprime pair $1 < m < n$ with $mn \le 600$ at once |
| $\sigma_{11}(n) \bmod 691$ | `sigma.index_add_(0, d * j, power[d - 1])` | (N + 1,) | every divisor pair $(d, j)$ with $dj \le N$ adds $d^{11}$ onto $n = dj$ |
| $x_p$ | `tau[p] / (2 * p ** 5.5)` | (P,) | normalised values at the primes, in $[-1, 1]$ by Deligne's theorem |

Ramanujan (1916) studied the coefficients $\tau(n)$ of $\Delta(q) = q\prod(1-q^n)^{24}$, the unique normalised cusp form of weight 12, and conjectured three things: $\tau$ is multiplicative, it satisfies $\tau(p^{k+1}) = \tau(p)\tau(p^k) - p^{11}\tau(p^{k-1})$, and $|\tau(p)| \le 2p^{11/2}$. Mordell proved the first two in 1917, an early case of what became Hecke theory. The bound is the Ramanujan conjecture for $\Delta$ (Ramanujan–Petersson in general): Deligne proved it in 1974 as a consequence of his proof of the Weil conjectures, so it is a theorem. Ramanujan also proved the congruence $\tau(n) \equiv \sigma_{11}(n) \pmod{691}$, which comes from $691$ dividing the numerator of the Bernoulli number $B_{12}$. Writing $\tau(p) = 2p^{11/2}\cos\theta_p$, the Sato–Tate conjecture says that the angles $\theta_p$ follow $\frac{2}{\pi}\sin^2\theta\,d\theta$, that is $x_p = \cos\theta_p$ follows the semicircle $\frac{2}{\pi}\sqrt{1-x^2}$, with moments $1/4$, $1/8$, $5/64$ (Catalan numbers over powers of 4). For $\Delta$ this was proved by Barnet-Lamb, Geraghty, Harris and Taylor (2011). The histogram converges slowly, like a sample of independent draws. Lehmer's conjecture that $\tau(n)$ is never zero is still open; it has been verified far beyond this range, and the program finds no zero either. Numerics: $\tau(n)$ grows like $n^{11/2}$, so a float64 FFT of the whole series has an absolute error of about $10^{-16}$ times the largest coefficient, which swamps $\tau(2) = -24$ (a test shows it). Instead the series is computed exactly modulo ten or so primes at once, one tensor row each, and rebuilt by the Chinese remainder theorem. Fun: Ramanujan noticed that… $\tau(n) \equiv \sigma_{11}(n) \pmod{691}$; that $1729 = 1^3 + 12^3 = 9^3 + 10^3$; and, in a regularised sense only, that $1 + 2 + 3 + \dots$ "$= -1/12$": the series diverges, and $-1/12$ is the value $\zeta(-1)$ of the analytically continued zeta function.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/ramanujan_tau/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n_max = 20000 if fast_mode() else params["n_max"]
    n_exact = 600

    # 1. exact τ(n) in int64 from Δ = q Π (1 − qⁿ)²⁴
    tau_e = solve.tau_exact(n_exact, dev)
    print("τ(1 … 12) =", ", ".join(f"{t:,}" for t in tau_e[1:13].tolist()))
    m, n, ok = equation.multiplicativity(tau_e)
    lhs, rhs = equation.hecke(tau_e, tensors.primes_up_to(n_exact, dev))
    print(
        f"multiplicative: τ(mn) = τ(m)τ(n) on {int(ok.sum())}/{len(ok)} coprime pairs with mn ≤ {n_exact}; "
        f"Hecke recurrence on {int((lhs == rhs).sum())}/{len(lhs)} prime powers"
    )
    print(
        f"Ramanujan noticed that τ(mn) = τ(m)τ(n) for coprime m, n — checked on {len(ok):,} pairs (Mordell proved it)"
    )

    # 2. large n: exact residues modulo a batch of primes, Chinese remainder theorem
    moduli = tensors.crt_moduli(n_max, dev)
    tau = solve.tau_float(n_max, dev)
    print(f"τ({n_max:,}) = {tau[-1]:.6e}  (residues modulo {len(moduli)} primes below {moduli[-1] + 1:,})")
    agree = torch.allclose(tau[1 : n_exact + 1], tau_e[1:].double(), rtol=1e-13, atol=0)
    print(f"  agrees with the int64 series for n ≤ {n_exact}: {agree}")
    t691 = solve.tau_mod(n_max, [691], dev)[0]
    sigma = equation.sigma_mod(n_max, 11, 691, dev)
    same = bool(torch.equal(t691[1:], sigma[1:]))
    print(f"Ramanujan noticed that τ(n) ≡ σ₁₁(n) (mod 691) — {'true' if same else 'FALSE'} for every n ≤ {n_max:,}")
    zeros = int((tau[1:] == 0).sum())
    print(f"Lehmer's conjecture (open): τ(n) ≠ 0 — {zeros} zeros for n ≤ {n_max:,}")

    # 3. Deligne's bound and the Sato–Tate law at the primes
    primes = tensors.primes_up_to(n_max, dev)
    x = equation.normalise(tau[primes], primes)  # (P,)
    worst = int(x.abs().argmax())
    print(
        f"Deligne: |τ(p)| ≤ 2p^(11/2) for all {len(primes):,} primes p ≤ {n_max:,}; "
        f"closest: p = {int(primes[worst]):,}, "
        f"|x_p| = {x.abs().max():.5f}"
    )
    target = equation.semicircle_moments(3)
    print(f"{'moment':>8} {'measured':>9} {'Sato–Tate':>10}")
    print(f"{'E[x]':>8} {x.mean():>9.4f} {0:>10.4f}")
    for k in range(3):
        print(f"{f'E[x^{2 * k + 2}]':>8} {(x ** (2 * k + 2)).mean():>9.4f} {target[k]:>10.4f}")

    data = {"p": primes, "x": x, "target": target}
    return plot.animate(data, out_dir / "ramanujan_tau.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/ramanujan_tau/equation.py
r"""Mathematics → tensor translation: Ramanujan's τ(n) as the coefficients of a power series.

1. The discriminant modular form

        Δ(q) = q Π_{n≥1} (1 − qⁿ)²⁴ = Σ_{n≥1} τ(n) qⁿ.

   Starting from Jacobi's sparse series for Π (1 − qⁿ)³, three squarings give the 6th, 12th and 24th powers. A
   truncated product c_n = Σ_j a_j b_{n−j} (repeated j ⇒ summed) is done three ways: as a Toeplitz matrix times a
   vector in int64 (exact, small n), by FFT in float64 (large n) and by FFT followed by rounding modulo m (exact
   residues).

2. Hecke theory (Mordell 1917): τ(mn) = τ(m) τ(n) for gcd(m, n) = 1 and
   τ(p^{k+1}) = τ(p) τ(p^k) − p¹¹ τ(p^{k−1}).

3. Ramanujan's congruence τ(n) ≡ σ₁₁(n) (mod 691), σ₁₁(n) = Σ_{d|n} d¹¹.

4. Deligne (1974): |τ(p)| ≤ 2 p^{11/2}, so x_p = τ(p) / (2 p^{11/2}) ∈ [−1, 1]. Sato–Tate (proved for Δ in 2011):
   the x_p are distributed with density (2/π) √(1 − x²), whose even moments are the Catalan numbers C_k / 4^k.
"""

import math

import torch


def mul_exact(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Truncated product c_n = Σ_{j≤n} a_{n−j} b_j as a Toeplitz matrix times a vector, int64.   (L,), (L,) → (L,)"""
    n = len(a)
    idx = torch.arange(n, device=a.device)
    lag = idx[:, None] - idx[None, :]  # (L, L): n − j
    toeplitz = torch.where(lag >= 0, a[lag.clamp(min=0)], 0)
    return toeplitz @ b


def mul_float(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """The same truncated product by FFT along the last axis (zero-padded against wrap-around), float64.
    (..., L), (..., L) → (..., L)"""
    n = a.shape[-1]
    spec = torch.fft.rfft(a.double(), 2 * n) * torch.fft.rfft(b.double(), 2 * n)
    return torch.fft.irfft(spec, 2 * n)[..., :n]


def mul_mod(a: torch.Tensor, b: torch.Tensor, m: torch.Tensor | int) -> torch.Tensor:
    """Truncated products of residues, one row per modulus: FFT, round to the exact integer, reduce.
    (M, L), (M, L), m (M, 1) → (M, L) int64

    Exact while the true sums (< m² L) stay far below 2^53, where float64 still counts in steps of one; the moduli
    are chosen with m² L ≤ 2^42, which leaves the FFT rounding error near 10^−2."""
    return mul_float(a, b).round().long() % m


def garner_balanced(residues: torch.Tensor, m: list[int]) -> torch.Tensor:
    """Chinese remainder theorem for every n at once, with balanced digits −m_i/2 < v_i ≤ m_i/2 so that negative
    numbers come out negative:  x = v_0 + v_1 m_0 + v_2 m_0 m_1 + …   (N, M) → (N, M) int64"""
    v = residues.clone()
    v[:, 0] = torch.where(v[:, 0] > m[0] // 2, v[:, 0] - m[0], v[:, 0])
    for i in range(1, len(m)):
        for j in range(i):
            v[:, i] = (v[:, i] - v[:, j]) % m[i] * pow(m[j], -1, m[i]) % m[i]
        v[:, i] = torch.where(v[:, i] > m[i] // 2, v[:, i] - m[i], v[:, i])
    return v


def digits_to_float(v: torch.Tensor, m: list[int]) -> torch.Tensor:
    """x = v_0 + m_0 (v_1 + m_1 (v_2 + …)) by Horner's rule in float64.   (N, M) → (N,)"""
    x = v[:, -1].double()
    for i in range(v.shape[1] - 2, -1, -1):
        x = v[:, i].double() + m[i] * x
    return x


def multiplicativity(tau: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Every coprime pair 1 < m < n with mn ≤ N, and whether τ(mn) = τ(m) τ(n).   (N + 1,) → (P,), (P,), (P,) bool"""
    n_max = len(tau) - 1
    r = torch.arange(2, n_max // 2 + 1, device=tau.device)
    m, n = torch.meshgrid(r, r, indexing="ij")
    keep = (m < n) & (m * n <= n_max) & (torch.gcd(m, n) == 1)
    m, n = m[keep], n[keep]
    return m, n, tau[m * n] == tau[m] * tau[n]


def hecke(tau: torch.Tensor, primes: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Both sides of τ(p^{k+1}) = τ(p) τ(p^k) − p¹¹ τ(p^{k−1}) for every prime power p^{k+1} ≤ N.
    (N + 1,) int64, (P,) → lhs (H,), rhs (H,)"""
    n_max = len(tau) - 1
    k = torch.arange(1, 64, device=tau.device)
    p, k = torch.meshgrid(primes.double(), k.double(), indexing="ij")
    keep = (k + 1) * torch.log(p) <= math.log(n_max) + 1e-9  # p^{k+1} ≤ N, decided before any power can overflow
    p, k = p[keep].long(), k[keep].long()
    p, k = p[p ** (k + 1) <= n_max], k[p ** (k + 1) <= n_max]
    lhs = tau[p ** (k + 1)]
    rhs = tau[p] * tau[p**k] - p**11 * tau[p ** (k - 1)]
    return lhs, rhs


def sigma_mod(n_max: int, k: int, mod: int, device: torch.device) -> torch.Tensor:
    """σ_k(n) = Σ_{d|n} d^k mod `mod` for n = 0 … n_max (σ_k(0) := 0).   → (N + 1,) int64

    Every pair (d, j) with d j ≤ N is listed once (about N ln N pairs) and d^k is added onto n = d j."""
    d_all = torch.arange(1, n_max + 1, device=device)
    power = torch.ones_like(d_all)
    for _ in range(k):
        power = power * d_all % mod  # d^k mod `mod`
    count = n_max // d_all  # multiples of d up to N
    d = torch.repeat_interleave(d_all, count)
    start = torch.cumsum(count, 0) - count
    j = torch.arange(len(d), device=device) - torch.repeat_interleave(start, count) + 1
    sigma = torch.zeros(n_max + 1, dtype=torch.int64, device=device)
    sigma.index_add_(0, d * j, power[d - 1])
    return sigma % mod


def normalise(tau_p: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    """x_p = τ(p) / (2 p^{11/2}), in [−1, 1] by Deligne's theorem.   (P,) → (P,)"""
    return tau_p.double() / (2 * p.double() ** 5.5)


def semicircle(x: torch.Tensor) -> torch.Tensor:
    """Sato–Tate density (2/π) √(1 − x²) on [−1, 1]."""
    return 2 / math.pi * torch.sqrt((1 - x**2).clamp(min=0))


def semicircle_moments(k_max: int) -> torch.Tensor:
    """E[x^{2k}] = C_k / 4^k for k = 1 … k_max (C_k the Catalan numbers).   → (K,)"""
    return torch.tensor([math.comb(2 * k, k) / (k + 1) / 4**k for k in range(1, k_max + 1)], dtype=torch.float64)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/ramanujan_tau/tensors.py
"""Build the concrete tensors: Jacobi's sparse series for Π (1 − qⁿ)³ and the primes."""

import math

import torch


def eta_cubed(length: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    """Coefficients of Π_{n≥1} (1 − qⁿ)³ up to q^{length − 1} from Jacobi's identity:

        Π (1 − qⁿ)³ = Σ_{k≥0} (−1)^k (2k + 1) q^{k(k+1)/2},

    so only about √(2·length) coefficients are nonzero.   → (length,)"""
    k = torch.arange(0, math.isqrt(2 * length) + 2, device=device)
    pos = k * (k + 1) // 2  # triangular numbers 0, 1, 3, 6, 10, …
    val = torch.where(k % 2 == 0, 1, -1) * (2 * k + 1)
    keep = pos < length
    series = torch.zeros(length, dtype=dtype, device=device)
    series[pos[keep]] = val[keep].to(dtype)
    return series


def primes_up_to(n: int, device: torch.device) -> torch.Tensor:
    """All primes ≤ n by the sieve of Eratosthenes.   → (P,) int64"""
    is_prime = torch.ones(n + 1, dtype=torch.bool, device=device)
    is_prime[:2] = False
    for p in range(2, math.isqrt(n) + 1):
        if is_prime[p]:
            is_prime[p * p :: p] = False
    return torch.nonzero(is_prime).flatten()


def crt_moduli(n_max: int, device: torch.device) -> list[int]:
    """Primes m with m² · n_max ≤ 2^42 (so FFT products of residues round exactly), enough of them that their product
    exceeds 2 |τ(n)| for every n ≤ n_max. Deligne: |τ(n)| ≤ d(n) n^{11/2}, and the divisor count d(n) ≤ 2√n."""
    cap = math.isqrt(2**42 // n_max)
    bits = 6 * math.log2(n_max) + 4  # log₂(2 · 2√n · n^{11/2}) plus a margin
    count = math.ceil(bits / math.log2(cap)) + 1
    return primes_up_to(cap, device)[-count:].tolist()


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/ramanujan_tau/solve.py
"""Expand Δ(q) = q Π (1 − qⁿ)²⁴: (η³)² = η⁶, (η⁶)² = η¹², (η¹²)² = η²⁴, then shift by one power of q."""

from collections.abc import Callable

import torch

from . import tensors
from .equation import digits_to_float, garner_balanced, mul_exact, mul_mod


def _delta(n_max: int, seed: torch.Tensor, mul: Callable[[torch.Tensor, torch.Tensor], torch.Tensor]) -> torch.Tensor:
    """τ(0), …, τ(n_max) from the series of Π (1 − qⁿ)³ (length n_max) and a truncated product."""
    e = seed
    for _ in range(3):
        e = mul(e, e)  # exponent 3 → 6 → 12 → 24
    return torch.cat(
        [e.new_zeros(*e.shape[:-1], 1), e], dim=-1
    )  # the factor q: τ(n) is the coefficient of q^{n−1} in Π (1 − qⁿ)²⁴


def tau_exact(n_max: int, device: torch.device) -> torch.Tensor:
    """Exact τ(n) in int64 by Toeplitz products (fine up to n ≈ 1000).   → (N + 1,) int64"""
    return _delta(n_max, tensors.eta_cubed(n_max, device, torch.int64), mul_exact)


def tau_mod(n_max: int, moduli: list[int], device: torch.device) -> torch.Tensor:
    """Exact residues of τ(n) modulo several m at once, one row each, by FFT products with rounding.
    → (M, N + 1) int64"""
    m = torch.tensor(moduli, device=device)[:, None]  # (M, 1)
    seed = tensors.eta_cubed(n_max, device, torch.int64)[None, :] % m  # (M, N)
    return _delta(n_max, seed, lambda a, b: mul_mod(a, b, m))


def tau_float(n_max: int, device: torch.device) -> torch.Tensor:
    """τ(n) for large n: exact residues modulo a batch of primes, then the Chinese remainder theorem, read out in
    float64 (so every value is correct to float64 rounding, even where τ(n) ≈ 10^30).   → (N + 1,) float64"""
    moduli = tensors.crt_moduli(n_max, device)
    residues = tau_mod(n_max, moduli, device)  # (M, N + 1)
    return digits_to_float(garner_balanced(residues.T, moduli), moduli)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/ramanujan_tau/plot.py
"""The Sato–Tate histogram of x_p = τ(p)/(2p^{11/2}) filling up against the semicircle, inside Deligne's bound."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif

N_FRAMES = 40


def animate(d: dict, path: Path) -> Path:
    p = d["p"].cpu().numpy()
    x = d["x"].cpu().numpy()
    target = d["target"].cpu().numpy()
    count = np.arange(1, len(x) + 1)
    m2 = np.cumsum(x**2) / count  # running moments over the first n primes
    m4 = np.cumsum(x**4) / count

    fig, (ax_h, ax_s, ax_m) = plt.subplots(1, 3, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1.3, 1.2, 1]})

    bins = np.linspace(-1, 1, 41)
    centres = 0.5 * (bins[1:] + bins[:-1])
    xs = np.linspace(-1, 1, 400)
    bars = ax_h.bar(
        centres, np.zeros_like(centres), width=bins[1] - bins[0], color="#93c5fd", edgecolor="#1e40af", lw=0.5
    )
    ax_h.plot(xs, 2 / np.pi * np.sqrt(1 - xs**2), color=ACCENT[1], lw=2, label=r"Sato–Tate $\frac{2}{\pi}\sqrt{1-x^2}$")
    ax_h.set_xlim(-1.05, 1.05)
    ax_h.set_ylim(0, 1.05)
    ax_h.set_xlabel(r"$x_p = \tau(p)\,/\,2p^{11/2}$")
    ax_h.set_ylabel("density")
    ax_h.legend(fontsize=8, loc="upper right")
    title_h = ax_h.set_title("", fontsize=10)

    sc = ax_s.scatter(p[:1], x[:1], s=2, color=ACCENT[0], lw=0)
    for y in (-1, 1):
        ax_s.axhline(y, color=ACCENT[1], lw=1.2, ls="--")
    ax_s.text(p[-1], 1.04, "Deligne bound", ha="right", va="bottom", fontsize=8, color=ACCENT[1])
    ax_s.set_xscale("log")
    ax_s.set_xlim(2, p[-1] * 1.05)
    ax_s.set_ylim(-1.15, 1.2)
    ax_s.set_xlabel("prime $p$")
    ax_s.set_ylabel("$x_p$")
    ax_s.set_title(r"$|\tau(p)| \leq 2p^{11/2}$ (Deligne 1974)", fontsize=10)

    (l2,) = ax_m.semilogx(count[:1], m2[:1], color=ACCENT[0], lw=1.5, label=r"$\langle x^2\rangle$")
    (l4,) = ax_m.semilogx(count[:1], m4[:1], color=ACCENT[2], lw=1.5, label=r"$\langle x^4\rangle$")
    ax_m.axhline(target[0], color=ACCENT[0], lw=1, ls=":", label="1/4")
    ax_m.axhline(target[1], color=ACCENT[2], lw=1, ls=":", label="1/8")
    ax_m.set_xlim(1, len(x))
    ax_m.set_ylim(0, 0.45)
    ax_m.set_xlabel("number of primes")
    ax_m.legend(fontsize=8, loc="upper right", ncol=2)
    ax_m.set_title("moments → semicircle values", fontsize=10)

    fig.text(
        0.5,
        0.01,
        "Ramanujan noticed that… |τ(p)| ≤ 2p^(11/2) (1916, conjectured). Deligne proved it in 1974.",
        ha="center",
        fontsize=9,
        color="#6b7280",
        style="italic",
    )

    def update(f: int):
        s = (f + 1) / N_FRAMES
        n = max(10, int(round(len(x) ** s)))
        h, _ = np.histogram(x[:n], bins=bins, density=True)
        for bar, height in zip(bars, h):
            bar.set_height(height)
        title_h.set_text(f"{n:,} primes up to {p[n - 1]:,}")
        sc.set_offsets(np.column_stack([p[:n], x[:n]]))
        l2.set_data(count[:n], m2[:n])
        l4.set_data(count[:n], m4[:n])

    fig.tight_layout(rect=(0, 0.04, 1, 1))
    return save_gif(fig, update, frame_indices(N_FRAMES, N_FRAMES), path, fps=6)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_max': 200000
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.ramanujan_tau")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.ramanujan_tau.main")
problem_dir = Path("problems/maths/ramanujan_tau")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
